# Week 6 — Parabolic & Hyperbolic PDEs: Heat and Wave Equations

> **Differential Equations for Scientists & Engineers**  
> *FTCS, Crank-Nicolson, and the leapfrog scheme — derived from scratch, stability verified.*

---

## Learning Objectives

1. Derive the **heat equation** from Fourier's law; separate variables analytically
2. Implement **FTCS** (Forward-Time, Centred-Space) and analyse the **CFL stability condition**
3. Implement **Crank-Nicolson** for the heat equation (A-stable, 2nd-order in time)
4. Derive the **wave equation** and its d'Alembert solution
5. Implement the **leapfrog (explicit centred)** scheme for the wave equation
6. Visualise PDE solutions as animated surfaces and space-time diagrams


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. The Heat Equation

Fourier's law of heat conduction leads to:

$$\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2}, \quad x \in (0, L), \quad t > 0$$

with initial condition $u(x, 0) = u_0(x)$ and boundary conditions $u(0,t) = u(L,t) = 0$.

**Analytical solution via separation of variables:**

$$u(x, t) = \sum_{n=1}^{\infty} B_n\,\sin\frac{n\pi x}{L}\,e^{-\alpha(n\pi/L)^2 t}$$

where $B_n = \dfrac{2}{L}\int_0^L u_0(x)\sin\dfrac{n\pi x}{L}\,dx$.

In [ ]:
def heat_analytical(x, t, alpha, L, N_terms=50):
    """Analytical Fourier series solution to the heat equation."""
    u = np.zeros_like(x, dtype=float)
    for n in range(1, N_terms+1):
        # B_n for u0(x) = sin(pi*x/L) is 1 if n=1, else 0
        # Let's use a more interesting IC: u0(x) = x*(L-x)
        # B_n = (2/L) * integral_0^L x(L-x)sin(n*pi*x/L)dx
        # = (8L^2) / (n^3*pi^3)  for odd n, 0 for even n
        if n % 2 == 1:
            B_n = 8 * L**2 / (n**3 * np.pi**3)
            lam_n = (n * np.pi / L)**2
            u += B_n * np.sin(n * np.pi * x / L) * np.exp(-alpha * lam_n * t)
    return u


L = 1.0; alpha = 0.01
x = np.linspace(0, L, 200)
times = [0.0, 0.5, 2.0, 5.0, 20.0, 50.0]

fig, ax = plt.subplots(figsize=(9, 5))
colors = cm.plasma(np.linspace(0.1, 0.9, len(times)))

for t_val, c in zip(times, colors):
    u = heat_analytical(x, t_val, alpha, L)
    ax.plot(x, u, color=c, lw=2, label=f't={t_val}')

ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
ax.set_title('Heat Equation — Analytical Solution ($u_0 = x(1-x)$)')
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); plt.show()

---

## 2. FTCS Scheme — Forward-Time Centred-Space

Discretise with $\Delta x = h$, $\Delta t = k$:

$$\frac{u_i^{n+1} - u_i^n}{k} = \alpha\,\frac{u_{i-1}^n - 2u_i^n + u_{i+1}^n}{h^2}$$

Defining $r = \alpha k/h^2$ (the **Fourier number** or mesh ratio):

$$\boxed{u_i^{n+1} = r\,u_{i-1}^n + (1-2r)\,u_i^n + r\,u_{i+1}^n}$$

**Von Neumann stability analysis** requires $r \leq 1/2$ for stability.

In [ ]:
def ftcs_heat(u0, alpha, L, T, Nx, Nt):
    """
    FTCS explicit scheme for u_t = alpha*u_xx.
    Returns solution array (Nt+1, Nx+1) and Fourier number r.
    """
    dx = L / Nx; dt = T / Nt
    r = alpha * dt / dx**2
    x = np.linspace(0, L, Nx+1)

    U = np.zeros((Nt+1, Nx+1))
    U[0] = u0(x)
    U[:, 0] = 0; U[:, -1] = 0  # Dirichlet BCs

    for n in range(Nt):
        U[n+1, 1:-1] = (r * U[n, :-2] +
                        (1 - 2*r) * U[n, 1:-1] +
                        r * U[n, 2:])
    return x, U, r


u0 = lambda x: x * (1 - x)   # IC: parabola

# Stable run (r < 0.5)
x_s, U_stable, r_s = ftcs_heat(u0, alpha=0.01, L=1, T=5, Nx=50, Nt=5000)
# Unstable run (r > 0.5)
x_u, U_unstable, r_u = ftcs_heat(u0, alpha=0.01, L=1, T=5, Nx=50, Nt=100)

print(f"Stable run:   r = {r_s:.4f} ({'<= 0.5 ✓' if r_s <= 0.5 else '> 0.5 ✗'})")
print(f"Unstable run: r = {r_u:.4f} ({'<= 0.5 ✓' if r_u <= 0.5 else '> 0.5 ✗'})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t_plot_idx = [0, 100, 500, 1000, 2000, 5000]
t_vals = np.linspace(0, 5, 5001)
colors = cm.viridis(np.linspace(0.1, 0.9, len(t_plot_idx)))

for idx, c in zip(t_plot_idx, colors):
    axes[0].plot(x_s, U_stable[idx], color=c, lw=1.5, label=f't={t_vals[idx]:.2f}')
axes[0].set_title(f'FTCS Stable (r={r_s:.3f} ≤ 0.5)')
axes[0].legend(frameon=False, fontsize=8)

for idx in [0, 10, 50, 100]:
    axes[1].plot(x_u, U_unstable[idx], lw=1, label=f't={5*idx/100:.2f}')
axes[1].set_title(f'FTCS Unstable (r={r_u:.3f} > 0.5) — Exponential blow-up')
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

---

## 3. Crank-Nicolson — Unconditionally Stable, 2nd-Order

Average the spatial operator between time levels $n$ and $n+1$:

$$\frac{u_i^{n+1} - u_i^n}{k} = \frac{\alpha}{2}\left(\delta^2_x u_i^n + \delta^2_x u_i^{n+1}\right)$$

This produces the tridiagonal system at each time step:

$$-\frac{r}{2}u_{i-1}^{n+1} + (1+r)u_i^{n+1} - \frac{r}{2}u_{i+1}^{n+1} = \frac{r}{2}u_{i-1}^n + (1-r)u_i^n + \frac{r}{2}u_{i+1}^n$$

Order: $O(k^2 + h^2)$. Stability: unconditional (A-stable).

In [ ]:
def thomas(a, b, c, d):
    """Thomas algorithm for tridiagonal systems."""
    n = len(b)
    c_ = np.zeros(n); d_ = np.zeros(n); x = np.zeros(n)
    c_[0] = c[0] / b[0]; d_[0] = d[0] / b[0]
    for i in range(1, n):
        denom = b[i] - a[i-1] * c_[i-1]
        c_[i] = (c[i] / denom) if i < n-1 else 0
        d_[i] = (d[i] - a[i-1] * d_[i-1]) / denom
    x[-1] = d_[-1]
    for i in range(n-2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i+1]
    return x


def crank_nicolson_heat(u0, alpha, L, T, Nx, Nt):
    """Crank-Nicolson scheme for u_t = alpha*u_xx."""
    dx = L / Nx; dt = T / Nt
    r = alpha * dt / dx**2
    x = np.linspace(0, L, Nx+1)
    M = Nx - 1  # interior points

    U = np.zeros((Nt+1, Nx+1))
    U[0] = u0(x)

    # Tridiagonal coefficients (constant)
    main_diag = (1 + r) * np.ones(M)
    off_diag  = (-r/2)  * np.ones(M-1)

    for n in range(Nt):
        un_int = U[n, 1:-1]
        rhs = (r/2 * np.pad(un_int, (0,1))[1:M+1] +   # right shift
               (1-r) * un_int +
               r/2 * np.pad(un_int, (1,0))[:M])         # left shift
        # BC contributions
        rhs[0]  += r/2 * U[n, 0]   + r/2 * U[n+1, 0]
        rhs[-1] += r/2 * U[n, -1]  + r/2 * U[n+1, -1]

        U[n+1, 1:-1] = thomas(off_diag, main_diag, off_diag, rhs)
    return x, U, r


# CN with large dt (r >> 0.5)
x_cn, U_cn, r_cn = crank_nicolson_heat(u0, alpha=0.01, L=1, T=5, Nx=50, Nt=100)
print(f"Crank-Nicolson: r = {r_cn:.2f} (unconditionally stable)")

fig, ax = plt.subplots(figsize=(9, 5))
colors = cm.plasma(np.linspace(0.1, 0.9, 6))
for idx, c in zip([0, 10, 25, 50, 75, 100], colors):
    ax.plot(x_cn, U_cn[idx], color=c, lw=2, label=f't={5*idx/100:.2f}')
ax.set_title(f'Crank-Nicolson (r={r_cn:.1f}, large time step, still stable!)')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 4. Wave Equation — Leapfrog Scheme

The 1D wave equation:

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2}$$

Centred differences in both time and space give the **leapfrog scheme**:

$$u_i^{n+1} = 2u_i^n - u_i^{n-1} + \nu^2\left(u_{i-1}^n - 2u_i^n + u_{i+1}^n\right)$$

where $\nu = c\Delta t/\Delta x$ is the **Courant number**. Stable if $\nu \leq 1$ (CFL condition).

In [ ]:
def leapfrog_wave(u0_func, du0_func, c, L, T, Nx, Nt):
    """
    Leapfrog (second-order centred) scheme for u_tt = c^2*u_xx.
    u0_func:  initial displacement u(x,0)
    du0_func: initial velocity  du/dt(x,0)
    """
    dx = L / Nx; dt = T / Nt
    nu = c * dt / dx
    x = np.linspace(0, L, Nx+1)

    U = np.zeros((Nt+1, Nx+1))
    U[0] = u0_func(x)

    # First time step via Taylor expansion: u^1 = u^0 + dt*ut^0 + dt^2/2 * u_tt^0
    u0 = U[0].copy()
    # u_tt^0 = c^2 * u_xx^0  (use centred diff)
    uxx0 = np.zeros(Nx+1)
    uxx0[1:-1] = (u0[:-2] - 2*u0[1:-1] + u0[2:]) / dx**2
    U[1, 1:-1] = (u0[1:-1] + dt * du0_func(x[1:-1]) +
                  0.5 * dt**2 * c**2 * uxx0[1:-1])

    for n in range(1, Nt):
        U[n+1, 1:-1] = (2 * U[n, 1:-1] - U[n-1, 1:-1] +
                        nu**2 * (U[n, :-2] - 2*U[n, 1:-1] + U[n, 2:]))
        U[n+1, 0] = U[n+1, -1] = 0  # Dirichlet

    return x, U, nu


# Gaussian pulse on [0,1]
c_wave = 1.0
u0_gauss = lambda x: np.exp(-200*(x - 0.3)**2)
du0_zero = lambda x: np.zeros_like(x)

x_w, U_wave, nu = leapfrog_wave(u0_gauss, du0_zero, c_wave, L=1, T=1.5, Nx=200, Nt=300)
print(f"Courant number nu = {nu:.3f} ({'<= 1 ✓' if nu <= 1 else '> 1 UNSTABLE ✗'})")

# Space-time plot (Hovmöller diagram)
t_wave = np.linspace(0, 1.5, 301)
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(x_w, t_wave, U_wave, cmap='RdBu_r',
                   vmin=-1, vmax=1, shading='auto')
plt.colorbar(im, ax=ax, label='u(x,t)')
ax.set_xlabel('x'); ax.set_ylabel('t')
ax.set_title(f'Wave Equation — Leapfrog (Courant ν={nu:.2f})')
plt.tight_layout(); plt.show()

---

## 5. Exercises

1. **(Heat equation)** Show that the analytical Fourier series solution satisfies the heat equation by differentiating term by term. Verify numerically that the FTCS error is $O(\Delta t + \Delta x^2)$.

2. **(Von Neumann stability)** Carry out the Von Neumann stability analysis for the FTCS scheme by substituting $u_j^n = \xi^n e^{i j k h}$ and find the amplification factor $\xi(k, r)$.

3. **(CN order verification)** Fix $\Delta x$ and vary $\Delta t$; confirm that Crank-Nicolson error decreases as $O(\Delta t^2)$. Then fix $\Delta t$ and vary $\Delta x$ to confirm $O(\Delta x^2)$.

4. **(d'Alembert)** The analytical solution to the wave equation is $u(x,t) = F(x-ct) + G(x+ct)$. Show that the Gaussian pulse IC splits into two waves propagating in opposite directions. Verify this against the leapfrog output.

5. **(2D heat equation)** Extend FTCS to 2D: $u_t = \alpha(u_{xx} + u_{yy})$ on the unit square with zero Dirichlet BCs and IC $u_0 = \sin(\pi x)\sin(\pi y)$. Implement it and verify against the analytical solution.